In [ ]:
#test_roi - experiments with defining regions of interest
#TODO: combine with test_proj and convert to production GD_Landsat_01_Setup for glacierPropsLandsat.csv
#Region of interest is defined by a box drawn on the map LonMin, LatMin, LonMax, LatMax. 
#Then the region is converted to UTM8N coordinates, turned into a rectangle aligned with the baseline there.
#Then converted back into WGS84 and stored as x1,y1 through x5,y5.
#Saved as glacierPropsLandsat.csv and in glacierPropsLandsat.shp
#Don't modify x,y manually - only adjust the Lon,Lat set.
#PROBLEM: not all Landsat use UTM8N, some use 7.
import geemap
import pandas as pd
import ee
import os
# Initialize Earth Engine
ee.Initialize()

In [ ]:
# Load the CSV file
df = pd.read_csv(r'C:\Users\andyb\Documents\U\SEAN_Glacier-Dynamics\glacierPropsLandsat.csv')

In [ ]:
# Initialize a geemap Map
Map = geemap.Map()

# Initialize an empty list to store features
features = []

# Loop through each region in the CSV and add it to the map
for index, row in df.iterrows():
    # Create a bounding box geometry
    region = ee.Geometry.Rectangle([row['LonMin'], row['LatMin'], row['LonMax'], row['LatMax']])
    # Create a feature with the region name
    feature = ee.Feature(region, {'name': row['Name']+" "+row['Region']})
    # Append to features list
    features.append(feature)
    # Add the region to the map with a unique color or style
    #Map.addLayer(feature, {'color': 'blue'}, row['Name']+" "+row['Region'])

In [ ]:
# Convert the list to a FeatureCollection
fc = ee.FeatureCollection(features)
#NOTE: plotting works if we overwrite fc with fc.style() but saving no longer works

# Apply the styling to the FeatureCollection
fcs = fc.style(**{ #asterisk passes in as arguments instead of dictionary
    'color': 'FF0000',      # Red outline
    'width': 2,             # 2 pixels wide
    'lineType': 'dotted',
    'fillColor': '00000000' # Fully transparent fill
    }) #(**style_params)

Map.addLayer(fcs, {}, 'All roi')
# Center the map on the Nth region
n=0
Map.centerObject(ee.Geometry.Rectangle([df['LonMin'][n], df['LatMin'][n], df['LonMax'][n], df['LatMax'][n]]), zoom=7)
# Display the map
Map

In [ ]:
type(fc)

In [ ]:
# Draw any shapes on the map using the Drawing tools before executing this code block
if Map.user_roi is not None:
    roi = Map.user_roi

In [ ]:
asdf=roi.getInfo()
#{'geodesic': False,
# 'type': 'Polygon',
# 'coordinates': [[[-137.147821, 58.823874], lower left
#   [-137.147821, 58.845905], upper left
#   [-137.090829, 58.845905], upper right
#   [-137.090829, 58.823874], lower right
#   [-137.147821, 58.823874]]]}
asdf['coordinates'][0][0][0]
#Want to be able to easily paste back into LonMin,LonMax,LatMin,LatMax
result = f"{asdf['coordinates'][0][0][0]:.3f},{asdf['coordinates'][0][2][0]:.3f},{asdf['coordinates'][0][0][1]:.3f},{asdf['coordinates'][0][1][1]:.3f}"
result

In [ ]:
#copy/paste result into glacierPropsLandsat.csv

In [ ]:
#export feature collection
out_shp = os.path.join(r'C:\Users\andyb\Documents\U\GEE-Courses\data', "glacierPropsLandsat.shp")
geemap.ee_export_vector(fc, out_shp, verbose=True)